# __3- Connecting The Camera__ 

The pipline :

→ Open Camera

→ Read frame

→ Prepare the frame

→ Pass the frame to the model

→ Make preduction

→ Write the predicted gusture on the screen


## 3.1 Import CV Libriry and Prepare Model For Preduction

In [ ]:
!pip install opencv-python

In [ ]:

import torch # pytorch main libirary
import torch.nn as nn 
import cv2  
from torchvision import transforms, models
from PIL import Image

# Load Pretrained ResNet-18 Model
model = models.resnet18(pretrained=True)

# Modify the Final Fully Connected Layer where we have 2 classes 
num_classes = 2

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_classes)
)

# The trained model 
model.load_state_dict(torch.load("best_model.pth", map_location='cpu'))
model.eval()

class_names = ["Open", "Close"]
 
# will used to prepare the frame before send it to model 
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 3.2 Open Camera and Start Immediate Preduction

In [24]:
import torch.nn.functional as F
from PIL import Image


img = Image.open(r"D:\Pictures\Camera Roll\WIN_20260225_01_59_24_Pro.jpg").convert("RGB")

img = test_transform(img)
img = img.unsqueeze(0)

with torch.no_grad():
    outputs = model(img)
    probs = F.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probs, 1)

print("Prediction:", class_names[predicted.item()])
print("Confidence:", confidence.item())


Prediction: Close
Confidence: 0.9999932050704956


In [27]:
cap = cv2.VideoCapture(1) # use OpenCV to open the camera 0 means the camera is in the same device

while True: # endless loop to read vedio continuously
    ret, frame = cap.read() # ret is bool value (does the frame token sucssfully?), frame is the capture itself
    
    if not ret: # if capturing the frame faild stop the loop
        break
    
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    frame = frame[:, w//3 : 3*w//3]  # نقص المنتصف

    # مهم جدًا: نحول BGR → RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # نحول إلى PIL
    img = Image.fromarray(rgb).convert("RGB")

    img = test_transform(img)
    img = img.unsqueeze(0)
    # transform the colors from BGR to RBG becuse OpenCV reads images in BGR
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    image = Image.fromarray(frame_rgb) # the transform needs the image in PIL

    input_tensor = test_transform(image).unsqueeze(0) # preprocessing the frame

    # because this is a test phase the gradients calculations should stop to speed up the process
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1) # converts the output to probabilities , softmax converts values ​​into probabilities between 0 and 1
        confidence, predicted = torch.max(probabilities,1) #choose highest probability

    gesture_name = class_names[predicted.item()] # change class num to name (0 -> fist , 1 -> palm)
    conf_value = confidence.item() # extracting the value of trust

    # write gesture name on the vedio
    cv2.putText(frame,
            f"{gesture_name} ({conf_value:.2f})", # the text that will be shown
            (20, 40), # the place
            cv2.FONT_HERSHEY_SIMPLEX, # text type
            1, # text size
            (0, 255, 0), # color
            2) # text thickness

    cv2.imshow("Hand Gesture Recognition", frame) # show the vedio
    

    
    if cv2.waitKey(1) & 0xFF == ord('q'): # to break the loop
        break

cap.release() # stop camera 
cv2.destroyAllWindows() # close all windows



unsqueeze(0) adds a new dimension so that the shape becomes: 

[1, 3, 224, 224]

Because the model accepts Batch even if it's just one image.

مشاكل الى الان:
- دائما يتوقع يد مفتوحة  

- يظهر توقع حتى لو مافي يد في الكاميرا
- يتردد كثير بين اليد المفتوحة والمغلقة

In [35]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=1,
                       min_detection_confidence=0.7)

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    frame = frame[:, w//3 : 3*w//3]

    # نحول RGB عشان MediaPipe
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    #   التحقق من وجود يد
    if results.multi_hand_landmarks:
        # فيه يد → نكمل التوقع
        image = Image.fromarray(frame_rgb)
        input_tensor = test_transform(image).unsqueeze(0)

        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probabilities, 1)

        gesture_name = class_names[predicted.item()]
        conf_value = confidence.item()

        text = f"{gesture_name} ({conf_value:.2f})"
        color = (0,255,0)

    else:
        text = "No Hand Detected"
        color = (0,0,255)

    cv2.putText(frame, text, (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow("Hand Gesture Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [36]:

import numpy as np

# فتح الكاميرا
cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    
    frame = cv2.flip(frame, 1)

   
    h, w, _ = frame.shape
    frame = frame[:, w//3 : 3*w//3]


    # كشف اليد بلون الجلد


    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    skin_pixels = cv2.countNonZero(mask)
    total_pixels = mask.size

   
    #  إذا فيه يد يرسلها للمودل

    if skin_pixels / total_pixels > 0.05:  # لو أكثر من 5% جلد

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame_rgb)

        input_tensor = test_transform(image).unsqueeze(0)

        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probabilities, 1)

        gesture_name = class_names[predicted.item()]
        conf_value = confidence.item()

        text = f"{gesture_name} ({conf_value:.2f})"
        color = (0, 255, 0)

    else:
        text = "No Hand Detected"
        color = (0, 0, 255)

    cv2.putText(frame,
                text,
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                color,
                2)

    cv2.imshow("Hand Gesture Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()